# BUNNY Ligand Descriptor Explorer with UMAP

This notebook downloads the Bidentate Unified N,N-Ligand Library (BUNNY) descriptor files from Zenodo,
allows you to select one of three datasets (free ligand, NiH2 complex, or NiF2 complex),
computes a UMAP embedding of the ligand descriptors, and generates an interactive Bokeh HTML visualization
where all ligands can be explored with hover tooltips.

In [33]:
import os
import sys
import zipfile
import tempfile
from pathlib import Path
from typing import Tuple, Dict, Optional

import numpy as np
import pandas as pd
import urllib.request
from urllib.error import URLError, HTTPError

# For UMAP
try:
    from umap import UMAP
    umap_available = True
except ImportError:
    umap_available = False
    print("Warning: UMAP not installed. Will attempt to install on demand.")

# For visualization
import bokeh
from bokeh.plotting import figure, output_file, save
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.layouts import column
from bokeh.palettes import Category10, Category20
from bokeh.transform import cumsum
import pandas.plotting._matplotlib.style as style

# For Zenodo API
import json

print(f"Loaded libraries successfully. UMAP available: {umap_available}")

Loaded libraries successfully. UMAP available: True


## Step 1: Download and Extract BUNNY Descriptors

Download the BUNNY descriptor ZIP from Zenodo and extract the three dataset files.

In [34]:
# Configuration
ZENODO_URL = "https://zenodo.org/api/records/19224101/files/Descriptor_libraries_BUNNY.zip/content"
EXTRACT_DIR = Path(tempfile.gettempdir()) / "bunny_descriptors"
ZIP_PATH = EXTRACT_DIR / "Descriptor_libraries_BUNNY.zip"

# Create extraction directory
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Download if not already present
if not ZIP_PATH.exists():
    print(f"Downloading BUNNY descriptors from Zenodo...")
    try:
        urllib.request.urlretrieve(ZENODO_URL, ZIP_PATH)
        print(f"Downloaded to {ZIP_PATH}")
    except (URLError, HTTPError) as e:
        print(f"Error downloading file: {e}")
        raise
else:
    print(f"ZIP file already exists at {ZIP_PATH}")

# Extract
print(f"Extracting to {EXTRACT_DIR}...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

# List extracted files
extracted_files = sorted(EXTRACT_DIR.glob("*.xlsx"))
print(f"\nExtracted {len(extracted_files)} datasets:")
for f in extracted_files:
    print(f"  - {f.name}")

ZIP file already exists at /var/folders/_g/6jml0x190szd85mnhhx81ydm0000gn/T/bunny_descriptors/Descriptor_libraries_BUNNY.zip
Extracting to /var/folders/_g/6jml0x190szd85mnhhx81ydm0000gn/T/bunny_descriptors...

Extracted 3 datasets:
  - metal_free_ligand_decriptors.xlsx
  - nif2_complex_ligand_descriptors.xlsx
  - nih2_complex_ligand_descriptors.xlsx


## Step 2: Select Dataset

Choose which ligand class to analyze: free ligand, NiH2 complex, or NiF2 complex.

In [35]:
# Define dataset mappings
dataset_mapping = {
    "Free Ligand": "metal_free_ligand_decriptors.xlsx",
    "NiH2 Complex": "nih2_complex_ligand_descriptors.xlsx",
    "NiF2 Complex": "nif2_complex_ligand_descriptors.xlsx",
}

# Select dataset (change this to select a different dataset)
selected_dataset_name = "Free Ligand"  # Options: "Free Ligand", "NiH2 Complex", "NiF2 Complex"
selected_file = dataset_mapping[selected_dataset_name]
dataset_path = EXTRACT_DIR / selected_file

print(f"Selected dataset: {selected_dataset_name}")
print(f"Loading from: {dataset_path}")

# Load the Excel file
df = pd.read_excel(dataset_path)
print(f"\nDataset shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()[:20]}...")  # Show first 20 columns
print(f"\nFirst few rows:")
print(df.iloc[:3, :5])  # Show first 3 rows, first 5 columns

Selected dataset: Free Ligand
Loading from: /var/folders/_g/6jml0x190szd85mnhhx81ydm0000gn/T/bunny_descriptors/metal_free_ligand_decriptors.xlsx

Dataset shape: (1068, 252)
Columns (252): ['id', 'SMILES', 'ligand_class', 'HOMO_min', 'HOMO_max', 'HOMO_avg', 'LUMO_min', 'LUMO_max', 'LUMO_avg', 'μ_min', 'μ_max', 'μ_avg', 'η_min', 'η_max', 'η_avg', 'ω_min', 'ω_max', 'ω_avg', 'polar_iso(Debye)_min', 'polar_iso(Debye)_max']...

First few rows:
        id                                             SMILES ligand_class  \
0   Lig723  C[Si](C)(C)N1C[C@]2(CN3CCC2CC3)N=C1C1=N[C@@]2(...         biim   
1   Lig769  C1CN2CCC1[C@]1(CN=C(C3=NC[C@@]4(CN5CCC4CC5)N3)...         biim   
2  Lig2145  CCCC(CCC)[C@@H]1N=C(C2=N[C@H](CN2C3=CC=CC=C3)C...         biim   

   HOMO_min  HOMO_max  
0  -0.20879  -0.20702  
1  -0.22037  -0.22037  
2  -0.21905  -0.21298  


## Step 3: Prepare Features for UMAP

Identify numeric descriptor columns and separate metadata (ligand IDs, class labels).
We'll exclude non-numeric columns and use the numeric descriptors for the embedding.

In [36]:
# Identify metadata and numeric columns
# Assume ligand ID is in the first column or a column with "ligand" or "id" in the name (case-insensitive)
metadata_col_patterns = ['ligand', 'id', 'name', 'code', 'class', 'type']

metadata_cols = []
numeric_cols = []

for col in df.columns:
    col_lower = col.lower()
    # Check if this looks like a metadata column
    if any(pattern in col_lower for pattern in metadata_col_patterns):
        metadata_cols.append(col)
    # Check if it's numeric
    elif pd.api.types.is_numeric_dtype(df[col]):
        numeric_cols.append(col)

print(f"Metadata columns ({len(metadata_cols)}): {metadata_cols}")
print(f"Numeric descriptor columns ({len(numeric_cols)}): {numeric_cols[:10]}...")

# Get the ligand identifier column (usually the first non-numeric column or first column)
ligand_id_col = metadata_cols[0] if metadata_cols else df.columns[0]
print(f"\nLigand ID column: {ligand_id_col}")

# Extract ligand class if available
ligand_class_col = None
for col in metadata_cols:
    if any(x in col.lower() for x in ['class', 'type', 'ligand type']):
        ligand_class_col = col
        break

if ligand_class_col:
    print(f"Ligand class column: {ligand_class_col}")
    print(f"Unique classes: {df[ligand_class_col].unique()[:10]}")
else:
    print("No ligand class column found. Will create default class.")
    ligand_class_col = None

# Prepare feature matrix (numeric columns only, remove NaN)
X = df[numeric_cols].fillna(df[numeric_cols].mean())
print(f"\nFeature matrix shape: {X.shape}")

Metadata columns (2): ['id', 'ligand_class']
Numeric descriptor columns (249): ['HOMO_min', 'HOMO_max', 'HOMO_avg', 'LUMO_min', 'LUMO_max', 'LUMO_avg', 'μ_min', 'μ_max', 'μ_avg', 'η_min']...

Ligand ID column: id
Ligand class column: ligand_class
Unique classes: ['biim' 'biox' 'box' 'pyox' 'pynx' 'bpy' 'phen']

Feature matrix shape: (1068, 249)


## Step 4: Compute UMAP Embedding

Use UMAP to reduce the high-dimensional descriptor space to 2D for visualization.
If UMAP is not installed, install it automatically.

In [37]:
# Install UMAP if not available
if not umap_available:
    print("Installing UMAP...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "umap-learn"])
    from umap import UMAP
    print("UMAP installed successfully!")

# Standardize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Computing UMAP embedding for {X_scaled.shape[0]} ligands with {X_scaled.shape[1]} features...")

# Compute UMAP
reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(X_scaled)

print(f"UMAP embedding shape: {embedding.shape}")
print(f"Embedding ranges: X [{embedding[:, 0].min():.2f}, {embedding[:, 0].max():.2f}], "
      f"Y [{embedding[:, 1].min():.2f}, {embedding[:, 1].max():.2f}]")

Computing UMAP embedding for 1068 ligands with 249 features...


/opt/homebrew/Caskroom/miniconda/base/envs/gp_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP embedding shape: (1068, 2)
Embedding ranges: X [-6.98, 19.09], Y [-6.65, 10.36]


## Step 5: Prepare Data for Visualization

Create a DataFrame with UMAP coordinates, ligand metadata, and class labels for coloring.

In [38]:
# Create visualization dataframe
viz_df = pd.DataFrame({
    'umap_x': embedding[:, 0],
    'umap_y': embedding[:, 1],
    'ligand_id': df[ligand_id_col].astype(str)
})

# Add ligand class if available, otherwise create a default
if ligand_class_col and ligand_class_col in df.columns:
    viz_df['ligand_class'] = df[ligand_class_col].astype(str)
else:
    # Try to infer class from ligand ID pattern (e.g., "Lig_biim", "Lig_bpy", etc.)
    # Extract the class identifier (usually after "Lig_" prefix)
    def extract_class(ligand_id):
        if '_' in str(ligand_id):
            parts = str(ligand_id).split('_')
            if len(parts) > 1:
                return '_'.join(parts[1:3])  # e.g., biim, bpy, phen
        return 'unknown'
    
    viz_df['ligand_class'] = viz_df['ligand_id'].apply(extract_class)

print("Visualization dataframe:")
print(viz_df.head(10))
print(f"\nLigand classes: {viz_df['ligand_class'].unique()}")
print(f"Class counts:\n{viz_df['ligand_class'].value_counts()}")

Visualization dataframe:
      umap_x     umap_y ligand_id ligand_class
0  11.150297   3.368547    Lig723         biim
1  10.145434   9.603806    Lig769         biim
2  11.646359   3.235966   Lig2145         biim
3  11.369541   3.697344   Lig2207         biim
4  11.071861   9.996096    Lig192         biox
5  10.953830  10.220646    Lig202         biox
6  10.677426  10.260817    Lig264         biox
7  10.623580   8.981118   Lig1977         biox
8  10.690227   8.909959   Lig1978         biox
9  11.117067   9.908900   Lig1995         biox

Ligand classes: ['biim' 'biox' 'box' 'pyox' 'pynx' 'bpy' 'phen']
Class counts:
ligand_class
bpy     602
pyox    153
biox    108
box     107
biim     68
pynx     15
phen     15
Name: count, dtype: int64


In [39]:
## Step 5b: Generate RDKit Molecule Images

# Generate 2D structure images for each ligand using RDKit and encode them as base64 PNG for embedding in Bokeh.


In [40]:
from rdkit import Chem
from rdkit.Chem import Draw
from io import BytesIO
import base64

# Get SMILES column (assuming it exists in the dataframe)
smiles_col = None
for col in df.columns:
    if col.lower() in ['smiles', 'smi']:
        smiles_col = col
        break

if smiles_col is None:
    print("Warning: No SMILES column found. Skipping molecule image generation.")
    viz_df['molecule_image'] = ""
else:
    print(f"Generating RDKit molecule images from SMILES column: {smiles_col}")
    
    # Function to generate base64-encoded PNG image from SMILES
    def smiles_to_image_base64(smiles, size=(200, 200)):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            # Generate 2D coordinates
            from rdkit.Chem import AllChem
            AllChem.Compute2DCoords(mol)
            # Draw molecule
            img = Draw.MolToImage(mol, size=size)
            # Convert to base64
            buffer = BytesIO()
            img.save(buffer, format='PNG')
            buffer.seek(0)
            img_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return img_base64
        except Exception as e:
            print(f"Error processing SMILES {smiles}: {e}")
            return None
    
    # Generate images for all ligands
    print("Processing ligands...")
    molecule_images = []
    for idx, smiles in enumerate(df[smiles_col]):
        if idx % 100 == 0:
            print(f"  {idx}/{len(df)} ligands processed")
        img_b64 = smiles_to_image_base64(str(smiles))
        molecule_images.append(img_b64)
    
    # Add images to viz_df
    viz_df['molecule_image'] = molecule_images
    print(f"✓ Generated {len([x for x in molecule_images if x is not None])} molecule images")


Generating RDKit molecule images from SMILES column: SMILES
Processing ligands...
  0/1068 ligands processed
  100/1068 ligands processed
  200/1068 ligands processed
  300/1068 ligands processed
  400/1068 ligands processed
  500/1068 ligands processed
  600/1068 ligands processed
  700/1068 ligands processed
  800/1068 ligands processed
  900/1068 ligands processed
  1000/1068 ligands processed
✓ Generated 1068 molecule images


## Step 6: Create Bokeh Interactive Visualization

Generate an interactive scatter plot colored by ligand class with hover tooltips showing ligand ID, class, and UMAP coordinates.

In [41]:
# Assign colors to classes
unique_classes = viz_df['ligand_class'].unique()
n_classes = len(unique_classes)

# Select color palette
if n_classes <= 10:
    colors = Category10[10][:n_classes]
elif n_classes <= 20:
    colors = Category20[20][:n_classes]
else:
    import colorsys
    colors = [colorsys.hsv_to_rgb(i / n_classes, 0.7, 0.9) for i in range(n_classes)]
    colors = ['#%02x%02x%02x' % (int(r*255), int(g*255), int(b*255)) for r, g, b in colors]

class_to_color = dict(zip(unique_classes, colors))
viz_df['color'] = viz_df['ligand_class'].map(class_to_color)

# Create Bokeh ColumnDataSource
source = ColumnDataSource(viz_df)

# Create figure
output_file(filename="bunny_ligand_explorer.html")

p = figure(
    width=900,
    height=700,
    title=f"BUNNY Ligand Descriptor Space - {selected_dataset_name}",
    toolbar_location="right",
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# Add scatter plot
scatter = p.scatter(
    'umap_x', 'umap_y',
    source=source,
    size=8,
    color='color',
    alpha=0.6,
    hover_color='red',
    hover_alpha=0.9
)

# Add hover tool with molecule images
from bokeh.models import HoverTool
hover_tooltips = [
    ("Ligand ID", "@ligand_id"),
    ("Class", "@ligand_class"),
    ("UMAP X", "@umap_x{0.00}"),
    ("UMAP Y", "@umap_y{0.00}"),
]

# Add molecule image if available
if 'molecule_image' in viz_df.columns and viz_df['molecule_image'].notna().any():
    hover_tooltips.append(("Molecule", '<img src="data:image/png;base64,@molecule_image" style="max-width:200px;max-height:200px">'))

hover = HoverTool(tooltips=hover_tooltips)
p.add_tools(hover)

# Customize plot
p.xaxis.axis_label = "UMAP 1"
p.yaxis.axis_label = "UMAP 2"
p.title.text_font_size = "16pt"

print(f"Bokeh plot created with {len(viz_df)} ligands and molecule images")

Bokeh plot created with 1068 ligands and molecule images


## Step 7: Create Legend and Add Labels

Add a legend to identify each ligand class by color.

In [42]:
# Add a legend by creating dummy glyphs for each class
from bokeh.models import Legend

legend_items = []
for class_name, color in class_to_color.items():
    # Add a dummy scatter point for legend purposes
    dummy_glyph = p.scatter([0], [0], size=8, color=color, alpha=0.6)
    legend_items.append((class_name, [dummy_glyph]))

# Create and add legend to the plot
if legend_items:
    legend = Legend(items=legend_items)
    legend.click_policy = "hide"  # Click to hide/show classes
    p.add_layout(legend, 'right')
    p.legend.location = "top_right"

print(f"Legend added with {len(legend_items)} classes")

Legend added with 7 classes


## Step 8: Export to HTML File

Save the Bokeh visualization as an interactive HTML file.

In [43]:
# Save the plot
output_path = Path.cwd() / "bunny_ligand_explorer.html"
output_file(filename=str(output_path))
save(p)

print(f"\n✓ Bokeh visualization saved to: {output_path}")
print(f"  Total ligands: {len(viz_df)}")
print(f"  Number of classes: {n_classes}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"\n  Open the file in a web browser to explore the ligands interactively!")
print(f"  - Hover over points to see ligand ID, class, and coordinates")
print(f"  - Use pan/zoom tools in the toolbar")
print(f"  - Click legend entries to hide/show ligand classes")


✓ Bokeh visualization saved to: /Users/julesschleinitz/Desktop/Code/BNNY/bunny_ligand_explorer.html
  Total ligands: 1068
  Number of classes: 7
  File size: 8282.9 KB

  Open the file in a web browser to explore the ligands interactively!
  - Hover over points to see ligand ID, class, and coordinates
  - Use pan/zoom tools in the toolbar
  - Click legend entries to hide/show ligand classes


## Step 9: Summary and Statistics

Display summary statistics and visualization information.

In [44]:
print("\n" + "="*60)
print("BUNNY LIGAND DESCRIPTOR VISUALIZATION - SUMMARY")
print("="*60)
print(f"\nDataset: {selected_dataset_name}")
print(f"  File: {selected_file}")
print(f"  Total ligands: {len(viz_df)}")
print(f"  Descriptor features: {X.shape[1]}")
print(f"\nLigand Classes:")
for class_name in sorted(unique_classes):
    count = (viz_df['ligand_class'] == class_name).sum()
    print(f"  - {class_name}: {count} ligands")

print(f"\nUMAP Embedding:")
print(f"  Dimensions: 2D (UMAP1, UMAP2)")
print(f"  Random state: 42 (reproducible)")
print(f"  Neighbors: 15")
print(f"  Min distance: 0.1")

print(f"\nVisualization Output:")
print(f"  Format: Interactive HTML (Bokeh)")
print(f"  File: bunny_ligand_explorer.html")
print(f"  Features:")
print(f"    - Hover tooltips (Ligand ID, Class, Coordinates)")
print(f"    - Pan and zoom tools")
print(f"    - Legend with click-to-hide functionality")
print(f"    - Color-coded by ligand class")

print("\n" + "="*60)


BUNNY LIGAND DESCRIPTOR VISUALIZATION - SUMMARY

Dataset: Free Ligand
  File: metal_free_ligand_decriptors.xlsx
  Total ligands: 1068
  Descriptor features: 249

Ligand Classes:
  - biim: 68 ligands
  - biox: 108 ligands
  - box: 107 ligands
  - bpy: 602 ligands
  - phen: 15 ligands
  - pynx: 15 ligands
  - pyox: 153 ligands

UMAP Embedding:
  Dimensions: 2D (UMAP1, UMAP2)
  Random state: 42 (reproducible)
  Neighbors: 15
  Min distance: 0.1

Visualization Output:
  Format: Interactive HTML (Bokeh)
  File: bunny_ligand_explorer.html
  Features:
    - Hover tooltips (Ligand ID, Class, Coordinates)
    - Pan and zoom tools
    - Legend with click-to-hide functionality
    - Color-coded by ligand class

